# Workshop 04 — Evaluate the FINE-TUNED Qwen3-4B + compare to base

**Run this after both:** notebook **01** (produced a fine-tuned model package) and notebook **03** (registered the eval dataset + reward function, and gave you a base-model baseline).

## What you'll do

1. Resolve the latest fine-tuned model package from notebook 01.
2. **Reuse** the dataset + reward function registered in notebook 03 (same names — no duplication).
3. Run a **Custom Scorer evaluation** with `evaluate_base_model=True`, so SageMaker scores the **fine-tuned and base models in one job** — side-by-side accuracy.
4. (Optional) MMLU on the fine-tuned model — *did fine-tuning erode general reasoning?*
5. (Optional) LLM-as-a-Judge on the fine-tuned model.

Because the dataset and scorer are identical to notebook 03, every number is directly comparable.

## Prerequisites

- Notebook 01 finished and registered a `ModelPackage` under `qwen3-4b-phish-sft`
- Notebook 03 ran (so the dataset `phreshphish-eval-100` and evaluator `phish-label-exact-match` exist)
- Same role / MLflow / Lambda-role config as notebook 03

## 0. Install / upgrade SDK

In [ ]:
%pip install -q --upgrade sagemaker datasets boto3 rich
# Restart the kernel after this cell.

## 1. Configuration

Keep these identical to notebook 03.

In [ ]:
HF_DATASET_ID = "gt2026workshop/phreshphish-2k"
HF_CONFIG_NAME = "text"
# S3 output — leave S3_BUCKET = None to use the account's DEFAULT SageMaker bucket
# (sagemaker-<region>-<account-id>, auto-created on first use). The session cell
# resolves S3_OUTPUT_PATH = s3://<bucket>/<prefix>/ once the session exists.
S3_BUCKET = None
S3_PREFIX = "qwen3-4b-phish-sft"
ROLE_ARN = None

BASE_MODEL = "huggingface-reasoning-qwen3-4b"
MODEL_PACKAGE_GROUP_NAME = "qwen3-4b-phish-sft"          # produced by notebook 01
FINETUNED_MODEL_PACKAGE_VERSION = None                   # None = latest version

# Must match notebook 03 exactly so we reuse the same registered resources
EVAL_DATASET_NAME = "phreshphish-eval-100"
REWARD_EVALUATOR_NAME = "phish-label-exact-match"

EVAL_LIMIT = 100
MLFLOW_RESOURCE_ARN = None  # auto-discovered below
JUDGE_MODEL = "anthropic.claude-3-5-haiku-20241022-v1:0"

## 2. SageMaker session + MLflow discovery

In [ ]:
import os, boto3
from sagemaker.core.helper.session_helper import Session
from rich import print as rprint
from rich.pretty import pprint

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)
sagemaker_session = Session(sagemaker_client=sm_client)

# Default SageMaker bucket (sagemaker-<region>-<acct>); override via S3_BUCKET above.
BUCKET = S3_BUCKET or sagemaker_session.default_bucket()
S3_OUTPUT_PATH = f"s3://{BUCKET}/{S3_PREFIX}/"


if ROLE_ARN is None:
    from sagemaker.core.helper.session_helper import get_execution_role
    ROLE_ARN = get_execution_role()


def discover_mlflow_app(session_client):
    try:
        apps = []
        for page in session_client.get_paginator("list_mlflow_apps").paginate():
            apps.extend(page.get("Summaries", []))
    except Exception as e:
        print(f"Could not list MLflow apps ({e}).")
        return None
    ready = [a for a in apps if a.get("Status") in ("Created", "Updated")]
    return ready[0]["Arn"] if ready else None


if MLFLOW_RESOURCE_ARN is None:
    MLFLOW_RESOURCE_ARN = discover_mlflow_app(sm_client)

rprint({"region": REGION, "role": ROLE_ARN, "output": S3_OUTPUT_PATH, "mlflow": MLFLOW_RESOURCE_ARN})

## 3. Resolve the fine-tuned model package

In [ ]:
resp = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=10,
)
summaries = resp.get("ModelPackageSummaryList", [])
if not summaries:
    raise RuntimeError(f"No model packages in group '{MODEL_PACKAGE_GROUP_NAME}'. Did notebook 01 finish?")

if FINETUNED_MODEL_PACKAGE_VERSION is None:
    pkg = summaries[0]
else:
    pkg = next((s for s in summaries if s["ModelPackageVersion"] == FINETUNED_MODEL_PACKAGE_VERSION), None)
    if pkg is None:
        raise RuntimeError(f"version {FINETUNED_MODEL_PACKAGE_VERSION} not found")

FINETUNED_MODEL_ARN = pkg["ModelPackageArn"]
rprint({"finetuned_model": FINETUNED_MODEL_ARN, "version": pkg.get("ModelPackageVersion"), "created": str(pkg.get("CreationTime"))})

## 4. Reuse the dataset + reward function from notebook 03

We fetch them by name. If notebook 03 wasn't run, this cell tells you to run it first (rather than silently re-creating divergent resources).

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.evaluator import Evaluator

try:
    eval_dataset = DataSet.get(name=EVAL_DATASET_NAME, sagemaker_session=sagemaker_session)
    phish_evaluator = Evaluator.get(name=REWARD_EVALUATOR_NAME, sagemaker_session=sagemaker_session)
except Exception as e:
    raise RuntimeError(
        f"Could not find '{EVAL_DATASET_NAME}' / '{REWARD_EVALUATOR_NAME}'. "
        f"Run notebook 03 first — it registers both. ({e})"
    )

rprint({"eval_dataset_arn": eval_dataset.arn, "evaluator_arn": phish_evaluator.arn})

## 5. Custom Scorer — fine-tuned vs base, one job

`evaluate_base_model=True` makes SageMaker score **both** the fine-tuned model package and the original base model on the same data, so you get a clean before/after row pair.

In [ ]:
from sagemaker.train.evaluate import CustomScorerEvaluator

ft_scorer = CustomScorerEvaluator(
    evaluator=phish_evaluator,
    model=FINETUNED_MODEL_ARN,         # fine-tuned model package
    dataset=eval_dataset.arn,
    s3_output_path=f"{S3_OUTPUT_PATH}eval/finetuned/custom-scorer/",
    evaluate_base_model=True,          # ← also re-score base for an in-job comparison
    mlflow_resource_arn=MLFLOW_RESOURCE_ARN,
    mlflow_experiment_name="qwen3-4b-phish-eval",
    mlflow_run_name="finetuned-custom-scorer",
)

ft_execution = ft_scorer.evaluate()
print("job:", ft_execution.name)

In [ ]:
ft_execution.wait(target_status="Succeeded", poll=15, timeout=3600)
print("final status:", ft_execution.status.overall_status)
ft_execution.show_results()

## 5b. Score from artifacts — fine-tuned vs base (the trustworthy numbers)

⚠️ As in notebook 03, the managed `label_exact_match` is unreliable for this reasoning model. We recompute both models' accuracy **directly from the inference artifacts**. Because we ran with `evaluate_base_model=True`, this single job wrote two inference files — one under the `EvaluateCustomModel` step (fine-tuned) and one under `EvaluateBaseModel` (base). We score both and show the lift.

In [ ]:
import re, ast, json, hashlib, boto3
from urllib.parse import urlparse
from collections import Counter

_THINK_CLOSED = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_THINK_OPEN   = re.compile(r"<think>.*", re.DOTALL | re.IGNORECASE)
_LABEL        = re.compile(r"(phish|benign)", re.IGNORECASE)
_URL          = re.compile(r"URL:\s*((?:https?://)?[^\s\\\"]+)")
_WS           = re.compile(r"\s+")

def extract_label(text):
    if not text:
        return ""
    try:
        v = ast.literal_eval(text)
        if isinstance(v, list) and v:
            text = str(v[0])
    except Exception:
        pass
    text = _THINK_CLOSED.sub(" ", text)
    text = _THINK_OPEN.sub(" ", text)
    m = _LABEL.search(text)
    return m.group(1).lower() if m else ""

def join_key(prompt):
    if not prompt:
        return None
    m = _URL.search(prompt)
    if m:
        return "url:" + m.group(1)
    body = prompt.split("---", 1)[-1] if "---" in prompt else prompt
    body = _WS.sub(" ", body).strip().lower()[:600]
    return "h:" + hashlib.sha1(body.encode("utf-8")).hexdigest()[:16]

def _split(uri):
    u = urlparse(uri); return u.netloc, u.path.lstrip("/")

def _read_jsonl(s3, bucket, key):
    data = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8")
    return [json.loads(l) for l in data.splitlines() if l.strip()]

def find_inference_output(s3, output_prefix_uri, step_substr=None):
    """Find inference_output.jsonl under a prefix. If step_substr is given
    (e.g. 'EvaluateCustomModel' or 'EvaluateBaseModel'), only match that step."""
    bucket, prefix = _split(output_prefix_uri)
    hits = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.rsplit("/", 1)[-1] != "inference_output.jsonl":
                continue
            if step_substr and step_substr not in key:
                continue
            hits.append(key)
    if not hits:
        raise FileNotFoundError(f"no inference_output.jsonl (step={step_substr}) under {output_prefix_uri} yet")
    return f"s3://{bucket}/{sorted(hits)[-1]}"

def score_from_artifacts(inference_s3_uri, ground_truth_s3_uri):
    s3 = boto3.client("s3")
    ib, ik = _split(inference_s3_uri); gb, gk = _split(ground_truth_s3_uri)
    truth = {}
    for r in _read_jsonl(s3, gb, gk):
        k = join_key(r.get("prompt", ""))
        if k:
            truth[k] = extract_label(r.get("completion", "")) or (r.get("completion") or "").strip().lower()
    matched = correct = no_label = unmatched = 0
    conf = Counter()
    for r in _read_jsonl(s3, ib, ik):
        k = join_key(r.get("prompt", "")); pred = extract_label(r.get("inference", ""))
        if k not in truth:
            unmatched += 1; continue
        matched += 1
        if not pred: no_label += 1
        conf[(truth[k], pred or "??")] += 1
        if pred == truth[k]: correct += 1
    confusion = {f"{t}>{p}": n for (t, p), n in conf.items()}
    return {"matched": matched, "unmatched": unmatched, "correct": correct,
            "no_label": no_label, "accuracy": correct / matched if matched else 0.0,
            "confusion": confusion}

def print_report(name, res):
    c = res["confusion"]; g = lambda t, p: c.get(f"{t}>{p}", 0)
    tp, tn, fp, fn = g("phish","phish"), g("benign","benign"), g("benign","phish"), g("phish","benign")
    print(f"\n{'='*46}\n{name}\n{'='*46}")
    print(f"  accuracy:  {res['accuracy']:.1%}  ({res['correct']}/{res['matched']})")
    if res["unmatched"]: print(f"  unmatched: {res['unmatched']}")
    if res["no_label"]:  print(f"  no_label:  {res['no_label']}")
    print("\n  confusion (rows=truth, cols=pred):")
    print("                pred=phish   pred=benign")
    print(f"    phish          {tp:>4}         {fn:>4}")
    print(f"    benign         {fp:>4}         {tn:>4}")
    pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    print(f"\n  phish  precision={pr:.1%}  recall={rc:.1%}  f1={f1:.1%}")

GROUND_TRUTH_S3 = f"{S3_OUTPUT_PATH}eval/eval.jsonl"   # staged by notebook 03
FT_OUTPUT_PREFIX = f"{S3_OUTPUT_PATH}eval/finetuned/custom-scorer/"
s3 = boto3.client("s3")

# This job ran evaluate_base_model=True → two steps wrote two inference files.
ft_inf   = find_inference_output(s3, FT_OUTPUT_PREFIX, step_substr="EvaluateCustomModel")
base_inf = find_inference_output(s3, FT_OUTPUT_PREFIX, step_substr="EvaluateBaseModel")
print("fine-tuned:", ft_inf)
print("base:      ", base_inf)

ft_metrics   = score_from_artifacts(ft_inf,   GROUND_TRUTH_S3)
base_metrics = score_from_artifacts(base_inf, GROUND_TRUTH_S3)
print_report("FINE-TUNED Qwen3-4B", ft_metrics)
print_report("BASE Qwen3-4B",        base_metrics)

lift = ft_metrics["accuracy"] - base_metrics["accuracy"]
print(f"\n{'#'*46}")
print(f"  LIFT from fine-tuning:  {lift:+.1%}  "
      f"({base_metrics['accuracy']:.0%} -> {ft_metrics['accuracy']:.0%})")
print(f"{'#'*46}")


### Read the comparison

`show_results()` prints `label_exact_match` for both models, but **trust section 5b's artifact-based numbers** (the managed metric is unreliable for this reasoning model). Expect the fine-tuned model (~95%+) to clearly beat base (~80%), mostly by fixing the false-negatives — phishing pages the base model waved through.

In [ ]:
# Locate the raw per-sample result JSON for a confusion-matrix / error analysis
from urllib.parse import urlparse

p = urlparse(S3_OUTPUT_PATH.rstrip("/"))
BUCKET, PREFIX = p.netloc, p.path.lstrip("/")
s3 = boto3.client("s3")
for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/eval/finetuned/custom-scorer/").get("Contents", []):
    if obj["Key"].endswith((".json", ".jsonl")):
        print(f"  {obj['Size']:>10,} B  s3://{BUCKET}/{obj['Key']}")

## 6. (Optional) MMLU — catastrophic-forgetting check

Compare against the base MMLU you ran in notebook 03. If the fine-tuned score is within ~2-3 points, the LoRA fine-tune didn't meaningfully damage general capability.

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks

Benchmark = get_benchmarks()

ft_mmlu = BenchMarkEvaluator(
    benchmark=Benchmark.MMLU,
    model=FINETUNED_MODEL_ARN,
    subtasks=["computer_security"],    # same subtask as notebook 03
    s3_output_path=f"{S3_OUTPUT_PATH}eval/finetuned/mmlu/",
    evaluate_base_model=True,          # base + fine-tuned in one job
    mlflow_resource_arn=MLFLOW_RESOURCE_ARN,
    mlflow_experiment_name="qwen3-4b-phish-eval",
    mlflow_run_name="finetuned-mmlu",
)
ft_mmlu_exec = ft_mmlu.evaluate()
print("job:", ft_mmlu_exec.name)

In [ ]:
ft_mmlu_exec.wait(target_status="Succeeded", poll=30, timeout=7200)
print("final status:", ft_mmlu_exec.status.overall_status)
ft_mmlu_exec.show_results()

## 7. (Optional) LLM-as-a-Judge on the fine-tuned model

Requires Bedrock access. Mirrors notebook 03's judge run so the two are comparable.

In [ ]:
import json
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

custom_metric = {
    "customMetricDefinition": {
        "name": "PhishingLabelAgreement",
        "instructions": (
            "You are evaluating a phishing-detection classifier. Given the input page summary in "
            "`prompt` and the model's prediction in `prediction`, decide whether the prediction's "
            "first meaningful word matches the correct label (phish or benign).\n\n"
            "Prompt: {{prompt}}\nPrediction: {{prediction}}"
        ),
        "ratingScale": [
            {"definition": "Agree", "value": {"floatValue": 1}},
            {"definition": "Disagree", "value": {"floatValue": 0}},
        ],
    }
}

ft_llmaj = LLMAsJudgeEvaluator(
    model=FINETUNED_MODEL_ARN,
    evaluator_model=JUDGE_MODEL,
    dataset=f"{S3_OUTPUT_PATH}eval/eval.jsonl",   # same JSONL nb 03 staged
    builtin_metrics=["Faithfulness"],
    custom_metrics=json.dumps([custom_metric]),
    s3_output_path=f"{S3_OUTPUT_PATH}eval/finetuned/llmaj/",
    evaluate_base_model=True,
    mlflow_resource_arn=MLFLOW_RESOURCE_ARN,
    mlflow_experiment_name="qwen3-4b-phish-eval",
    mlflow_run_name="finetuned-llmaj",
)
ft_llmaj_exec = ft_llmaj.evaluate()
print("job:", ft_llmaj_exec.name)

In [ ]:
ft_llmaj_exec.wait(target_status="Succeeded", poll=30, timeout=7200)
print("final status:", ft_llmaj_exec.status.overall_status)
ft_llmaj_exec.show_results()

## 8. Where to look

- **MLflow** (experiment `qwen3-4b-phish-eval`): every run from notebooks 03 and 04 in one place — `base-custom-scorer`, `finetuned-custom-scorer`, the MMLU and LLMAJ runs. Plot accuracy across them.
- **Studio model card → Evaluations tab**: all evaluations attached to the fine-tuned model package.
- **Raw JSON**: `s3://<bucket>/<prefix>/eval/finetuned/.../eval_results/`.

## 9. Clean up

Evaluation jobs are pay-per-run. The persistent resources are the registered dataset and the reward-function Lambda — delete if you won't reuse them:

```python
# phish_evaluator.delete()   # removes the evaluator (and its Lambda)
```